In [2]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from pprint import pprint

# ========================== CONFIG ==========================
COLLECTION_NAME = "python_coding_standards"
DB_PATH = "./chroma_python_standards_db2"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

def query_chroma(query_text: str, n_results: int = 5):
    """
    Query the Python Coding Standards knowledge base
    """
    # Load embedding function (same model used during ingestion)
    embedding_function = SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL)

    # Connect to existing ChromaDB
    client = chromadb.PersistentClient(path=DB_PATH)
    
    collection = client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_function
    )

    print(f"🔍 Querying: '{query_text}'")
    print(f"📊 Top {n_results} results:\n")

    # Perform similarity search
    results = collection.query(
        query_texts=[query_text],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]   # You can also include embeddings if needed
    )

    # Display results nicely
    for i in range(len(results['documents'][0])):
        print(f"{'='*80}")
        print(f"Result {i+1} | Score: {1 - results['distances'][0][i]:.4f} "
              f"(Distance: {results['distances'][0][i]:.4f})")
        print(f"Section   : {results['metadatas'][0][i]['section_title']}")
        print(f"Header Level : {results['metadatas'][0][i].get('header_level', 'N/A')}")
        print(f"Parent    : {results['metadatas'][0][i].get('parent_section', 'N/A')}")
        print("-" * 80)
        
        # Print first 500 characters of content (clean preview)
        content_preview = results['documents'][0][i][:500]
        if len(results['documents'][0][i]) > 500:
            content_preview += "..."
        print(content_preview)
        print()

    return results


# ========================== EXAMPLE QUERIES ==========================
if __name__ == "__main__":
    print("🤖 Python Coding Standards RAG Query Tool\n")

    example_queries = [
        "What are the naming conventions for classes and functions?",
        "What is the recommended maximum line length?",
        "How should I handle imports in Python files?",
        "Explain the error handling best practices",
        "What tools are recommended for code formatting and linting?",
        "Show me the Python coding workflow diagram",
        "What is the docstring format we should follow?",
        "How to write unit tests according to standards?"
    ]

    for idx, q in enumerate(example_queries, 1):
        print(f"\n{'#'*80}")
        print(f"Example Query {idx}: {q}")
        print(f"{'#'*80}")
        query_chroma(q, n_results=3)
        input("\nPress Enter for next query...")   # Remove this line if you don't want pause

🤖 Python Coding Standards RAG Query Tool


################################################################################
Example Query 1: What are the naming conventions for classes and functions?
################################################################################
🔍 Querying: 'What are the naming conventions for classes and functions?'
📊 Top 3 results:

Result 1 | Score: 0.5570 (Distance: 0.4430)
Section   : 3.3 Blank Lines
Header Level : 3
Parent    : 3.2 Maximum Line Length
--------------------------------------------------------------------------------
* Two blank lines between top-level functions and classes.
* One blank line between methods inside a class.

Result 2 | Score: 0.3152 (Distance: 0.6848)
Section   : Naming Conventions
Header Level : 2
Parent    : 3.4 String Quotes
--------------------------------------------------------------------------------
```ascii
| Type | Convention | Example |
| --- | --- | --- |
| Packages | lowercase | `mypackage` |
| Modules 